# 시계열·센서 EDA

각 Notebook은 독립 실행합니다. 기본값 DEMO=True는 합성 연습 데이터입니다. 실제 데이터는 설정 셀에서 DEMO=False와 경로·열·문제 유형을 지정하세요. 앞의 Notebook 실행이나 개인 모듈 설치가 필요하지 않습니다.

시간 예산은 모델 후보를 시작하기 전에 확인하는 소프트 제한입니다. 진행 중인 fit을 강제 중단하지 않습니다. 대회 지문과 공식 제출 규격을 우선합니다.

## 설정

long/wide, 단일/다중 개체, 등간격/불규칙 데이터를 구분합니다. 센서 구간 분류용 입력 형식은 별도 sensor Notebook을 참고하세요.

In [ ]:
DEMO=True
TRAIN_PATH='data/train.csv';TEST_PATH='data/test.csv'
TIME_COL='timestamp';ENTITY='series_id';VALUE='value';ID='id'
FREQUENCY='h' # pandas 주기. 등간격 예측에서 필수
LAGS=[1,2,3,24];ROLLING=[3,12,24]
VALID_FRACTION=.2;METRIC='rmse'
SAMPLE_PATH=None;TARGET_COLUMNS=['value'];OUTPUT='outputs/time_series/forecast.csv'


## 공통 함수

기본 로딩과 시각화

In [ ]:
import os, time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (accuracy_score, f1_score, log_loss, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score, classification_report,
    ConfusionMatrixDisplay, silhouette_score)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, IsolationForest
from sklearn.cluster import MiniBatchKMeans
from sklearn.dummy import DummyClassifier, DummyRegressor
SEED=42
rng=np.random.default_rng(SEED)

def read_table(path):
    p=Path(path)
    if not p.is_file(): raise FileNotFoundError(p)
    if p.suffix.lower()=='.parquet': return pd.read_parquet(p)
    return pd.read_csv(p, sep='\t' if p.suffix.lower()=='.tsv' else ',')

def check_ids(df, id_col):
    if id_col not in df or df[id_col].isna().any() or df[id_col].duplicated().any():
        raise ValueError(f'{id_col}: 각 예측 단위에 결측 없는 고유 ID가 필요합니다.')

def split_rows(df, target=None, task='classification', strategy='random', group=None, time_col=None, fraction=.25, gap=0):
    idx=np.arange(len(df))
    if not 0<fraction<1: raise ValueError('validation fraction은 0~1 사이여야 합니다.')
    if strategy=='group':
        if not group or df[group].isna().any(): raise ValueError('유효한 그룹 열이 필요합니다.')
        a,b=next(GroupShuffleSplit(n_splits=1,test_size=fraction,random_state=SEED).split(df,groups=df[group]))
        assert set(df.iloc[a][group]).isdisjoint(set(df.iloc[b][group]))
    elif strategy=='time':
        if not time_col: raise ValueError('시간 열을 지정하세요.')
        t=pd.to_datetime(df[time_col],errors='raise')
        if t.isna().any(): raise ValueError('시간 결측을 해결하세요.')
        unique=np.sort(t.unique());cut=int(len(unique)*(1-fraction))
        if cut<=gap or cut>=len(unique): raise ValueError('시간 분할에 필요한 데이터가 부족합니다.')
        a=idx[t<unique[cut-gap]];b=idx[t>=unique[cut]]
        assert t.iloc[a].max()<t.iloc[b].min()
    elif strategy in ('random','stratified'):
        strat=df[target] if task=='classification' and target else None
        if strat is not None and strat.value_counts().min()<2:
            raise ValueError('표본 1개인 클래스가 있습니다. 병합/수집/분할 정책을 검토하세요.')
        a,b=train_test_split(idx,test_size=fraction,random_state=SEED,stratify=strat)
    else: raise ValueError(f'지원하지 않는 분할: {strategy}')
    if not len(a) or not len(b): raise ValueError('빈 학습/검증 분할')
    if task=='classification' and target and set(df.iloc[b][target])-set(df.iloc[a][target]):
        raise ValueError('학습에 없는 클래스가 검증에 존재합니다.')
    return np.asarray(a),np.asarray(b)

def clean_tabular(X):
    out=X.copy()
    for c in out:
        if pd.api.types.is_numeric_dtype(out[c]):
            out[c]=pd.to_numeric(out[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
        else: out[c]=out[c].map(lambda v: str(v) if pd.notna(v) else np.nan)
    return out

def tabular_preprocessor(X, robust=False):
    num=X.select_dtypes(include='number').columns.tolist()
    cat=[c for c in X if c not in num]
    parts=[]
    if num: parts.append(('num',make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),RobustScaler() if robust else StandardScaler()),num))
    if cat: parts.append(('cat',make_pipeline(SimpleImputer(strategy='constant',fill_value='__MISSING__',keep_empty_features=True),OneHotEncoder(handle_unknown='ignore',min_frequency=2)),cat))
    if not parts: raise ValueError('특징 열이 없습니다.')
    return ColumnTransformer(parts)

def metrics_for(model,X,y,task):
    pred=model.predict(X)
    if task=='regression': return {'mae':float(mean_absolute_error(y,pred)),'rmse':float(np.sqrt(mean_squared_error(y,pred))),'r2':float(r2_score(y,pred))}
    out={'accuracy':float(accuracy_score(y,pred)),'f1_macro':float(f1_score(y,pred,average='macro',zero_division=0))}
    if hasattr(model,'predict_proba'):
        p=model.predict_proba(X);classes=model.classes_
        out['log_loss']=float(log_loss(y,p,labels=classes))
        if len(classes)==2 and len(np.unique(y))==2: out['roc_auc']=float(roc_auc_score(np.asarray(y)==classes[1],p[:,1]))
    return out

def fit_compare(candidates,X,y,a,b,task,metric,budget=120):
    direction={'accuracy':True,'f1_macro':True,'roc_auc':True,'r2':True,'log_loss':False,'mae':False,'rmse':False}
    if metric not in direction: raise ValueError('지원 지표를 선택하거나 metrics_for를 확장하세요.')
    t0=time.monotonic();rows=[];fitted={}
    for name,estimator in candidates.items():
        if rows and time.monotonic()-t0>=budget: break
        start=time.monotonic();model=clone(estimator).fit(X.iloc[a] if hasattr(X,'iloc') else X[a],y.iloc[a] if hasattr(y,'iloc') else y[a])
        stats=metrics_for(model,X.iloc[b] if hasattr(X,'iloc') else X[b],y.iloc[b] if hasattr(y,'iloc') else y[b],task)
        if metric not in stats or not np.isfinite(stats[metric]): raise ValueError(f'{metric}: 이 분할/모델에서 계산할 수 없습니다.')
        rows.append({'model':name,**stats,'seconds':time.monotonic()-start});fitted[name]=model
    table=pd.DataFrame(rows).sort_values(metric,ascending=not direction[metric]);display(table)
    best=table.iloc[0]['model']
    return best,fitted[best],table

def write_submission(ids,pred,id_col,target_cols,output,sample_path=None,probabilities=False):
    ids=pd.Series(ids).reset_index(drop=True)
    arr=np.asarray(pred)
    if arr.ndim==1: arr=arr[:,None]
    if arr.ndim!=2 or arr.shape!=(len(ids),len(target_cols)): raise ValueError('예측의 행/열 수와 제출 규격이 다릅니다.')
    if ids.isna().any() or ids.duplicated().any(): raise ValueError('제출 ID 결측/중복')
    if len(set(target_cols))!=len(target_cols) or id_col in target_cols: raise ValueError('제출 열 이름 중복')
    out=pd.DataFrame(arr,columns=target_cols);out.insert(0,id_col,ids.to_numpy())
    if out.isna().any().any(): raise ValueError('제출 값 결측')
    numeric=out[target_cols].select_dtypes(include='number')
    if numeric.size and not np.isfinite(numeric.to_numpy()).all(): raise ValueError('제출 값 무한대')
    if probabilities:
        v=arr.astype(float)
        if not np.isfinite(v).all() or (v<0).any() or (v>1).any(): raise ValueError('확률 범위 오류')
        if v.shape[1]>1 and not np.allclose(v.sum(axis=1),1,atol=1e-5): raise ValueError('확률 합 오류')
    if sample_path:
        sample=read_table(sample_path);check_ids(sample,id_col)
        if set(sample.columns)!=set(out.columns): raise ValueError('sample_submission과 열이 다릅니다.')
        if len(sample)!=len(out) or set(sample[id_col])!=set(out[id_col]): raise ValueError('sample_submission과 ID가 다릅니다. ID 자료형도 확인하세요.')
        out=sample[[id_col]].merge(out,on=id_col,how='left',validate='one_to_one')[sample.columns]
    p=Path(output);p.parent.mkdir(parents=True,exist_ok=True);out.to_csv(p,index=False)
    display(out.head());print('저장:',p,'형태:',out.shape)
    return out

def classification_output(model,X,kind='label',order=None,positive=None):
    if kind=='label': return model.predict(X)
    if not hasattr(model,'predict_proba'): raise ValueError('확률을 지원하는 모델이 필요합니다.')
    classes=list(model.classes_);p=model.predict_proba(X)
    if kind=='positive_probability':
        if positive not in classes: raise ValueError('positive_class를 실제 클래스 값으로 지정하세요.')
        return p[:,classes.index(positive)]
    if kind!='probability' or order is None or len(order)!=len(classes) or set(order)!=set(classes):
        raise ValueError('공식 제출 열에 대응하는 class_order를 지정하세요.')
    return p[:,[classes.index(v) for v in order]]


## 데이터 형식 확인

이 예측 예제는 timestamp·series_id·value long 형태를 사용합니다.

In [ ]:
if DEMO:
    records=[];future=[]
    for sid in ['A','B']:
        dates=pd.date_range('2025-01-01',periods=150,freq=FREQUENCY)
        values=5+np.sin(np.arange(150)*2*np.pi/24)+(sid=='B')*2+np.arange(150)*.01+rng.normal(0,.1,150)
        records.extend([{ENTITY:sid,TIME_COL:t,VALUE:v} for t,v in zip(dates,values)])
        future.extend([{ENTITY:sid,TIME_COL:t,ID:f'{sid}-{j}'} for j,t in enumerate(pd.date_range(dates[-1],periods=13,freq=FREQUENCY)[1:])])
    train=pd.DataFrame(records);test=pd.DataFrame(future)
else: train=read_table(TRAIN_PATH);test=read_table(TEST_PATH)
if not ENTITY:
    ENTITY='__series__';train[ENTITY]='single';test[ENTITY]='single'
for frame in [train,test]:
    frame[TIME_COL]=pd.to_datetime(frame[TIME_COL],errors='raise')
    if frame[[ENTITY,TIME_COL]].isna().any().any() or frame.duplicated([ENTITY,TIME_COL]).any(): raise ValueError('개체/시간 결측 또는 중복')
train[VALUE]=pd.to_numeric(train[VALUE],errors='raise')
print('값 결측:', train[VALUE].isna().sum(), '무한대:', np.isinf(train[VALUE]).sum())
train=train.sort_values([ENTITY,TIME_COL]).reset_index(drop=True)
check_ids(test,ID)
df=train


## 간격·추세·자기상관·주파수·형식 변환

FFT는 등간격일 때만 의미가 있습니다. 결측을 미래 값으로 보간하면 예측 누수가 생길 수 있습니다.

In [ ]:
for sid,g in train.groupby(ENTITY,sort=False):
    dt=g[TIME_COL].diff().dropna()
    print('series',sid,'rows',len(g),'간격 분포');display(dt.value_counts().head())
    fig,axes=plt.subplots(2,1,figsize=(10,5))
    axes[0].plot(g[TIME_COL],g[VALUE]);axes[0].set_title(str(sid))
    axes[1].plot(range(1,min(49,len(g))),[g[VALUE].autocorr(lag=k) for k in range(1,min(49,len(g)))]);axes[1].set_title('autocorrelation');plt.tight_layout();plt.show()
    if len(g)>8:
        values=g[VALUE].dropna().to_numpy();values=values-values.mean()
        if len(values)!=len(g) or g[TIME_COL].diff().dropna().nunique()!=1:
            print('FFT 생략: 결측 또는 불규칙 간격');continue
        freq=np.fft.rfftfreq(len(values),d=1)
        plt.plot(freq,abs(np.fft.rfft(values)));plt.xlabel('cycles/sample (등간격일 때만 해석)');plt.show()
    if train[ENTITY].nunique()>5: break
# wide -> long: 각 센서가 열인 표를 개체/시간/값 형태로 바꿀 때
wide=train.pivot(index=TIME_COL,columns=ENTITY,values=VALUE)
long=wide.reset_index().melt(id_vars=TIME_COL,var_name=ENTITY,value_name=VALUE)
display(long.head())
# 불규칙 간격: 과거 관측만 사용하는 ffill. 정답 예측 전에는 범위와 관측 가능 시점을 확인.
first_series=train[train[ENTITY]==train[ENTITY].iloc[0]].set_index(TIME_COL)
regular=first_series[[VALUE]].resample(FREQUENCY).last()
regular['missing_before_fill']=regular[VALUE].isna()
regular[VALUE]=regular[VALUE].ffill(limit=2)
display(regular.head())
